# Assignment 3 

## Group name: [enter your group name with the prefix "ID2214 -"]

### Project members: 
[Enter full name, email]
[Enter full name, email]
[Enter full name, email]

### Declaration:
By submitting this assignment, it is hereby declared that all group members listed above have contributed to the solution, either with code that appear in the final solution below, or with code that has been evaluated and compared to the final solution, but for some reason has been excluded. It is also declared that all project members fully understand all parts of the final solution and can explain it upon request.

It is furthermore declared that the code below is a contribution by the project members only, and specifically that no part of the solution has been copied from any other source (except for lecture slides at the course ID2214/FID3214), no part of the solution has been provided by someone not listed as a project member above, and no part of the solution has been generated by a system such as a Generative AI tool, *other than where the assignment text explicitly allows such use*. 

It is furthermore declared that the submitted assignment will not be shared during the course, with any individual other than the group members listed above and teachers of the course ID2214. In particular, the assignment will not be uploaded to any public repository. The submitted assignment can be shared after the course only if written consent has been provided by the course responsible of ID2214.

It is furthermore declared that it has been understood that no other library/package than the Python 3 standard library, NumPy and matplotlib may be used in the solution for this assignment.

### Instructions
All parts of the assignment starting with number 1 below are mandatory. Sufficient (but not necessarily correct) solutions
on these parts will give 1 point (in total). The remaining parts (starting with number 2) are optional and will not give additional points. 

In this assignment you will work with the monthly mean temperatures recorded in Stockholm since 1756. The models are
trained on the data up to and including 1989, are evaluated on the temperatures from 1990 until the last recorded year (2020)
and are extrapolated until 2045, both month by month and as a long-term trend over the decades. All hyperparameters are fixed in advance and are given as default
values of the functions below; no hyperparameter optimization is required.

Note that you do not have to develop the code directly within the notebook
but may instead copy the comments and test cases to a more convenient development environment
and when everything works as expected, you may paste your functions into this
notebook, do a final testing (all cells should succeed), save the notebook including the output
and submit the whole notebook (a single file) in Canvas; do not forget to fill in your group number
and names above, thereby confirming that you compy with the above declaration. 

**Read and confirm the declaration before starting!** 

## Load NumPy and matplotlib

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

In [ ]:
from platform import python_version

print(f"Python version: {python_version()}")
print(f"NumPy version: {np.__version__}")
print(f"matplotlib version: {matplotlib.__version__}")

## Load the data and create a training and a test set

In [ ]:
# Load the data (leave this part unchanged)
#
# The table is turned into a single time series where each month is placed at a decimal year.
#
# t, y             - time (in years) and temperature of all months
# t_train, y_train - all months before 1990 (training set)
# t_test, y_test   - all months from 1990 until the end (test set)
# y_train_c        - the training temperatures with their mean y_mean subtracted (the models assume zero-mean data)

data = np.loadtxt("stockholm_monthly_mean_temperature_1756_2020_adjust.csv",
                  delimiter=",", skiprows=1, usecols=range(13))
years, monthly = data[:, 0], data[:, 1:]

t = (years[:, None] + (np.arange(12) + 0.5) / 12).ravel()
y = monthly.ravel()

train = t < 1990
t_train, y_train = t[train], y[train]
t_test, y_test = t[~train], y[~train]

y_mean = y_train.mean()
y_train_c = y_train - y_mean

print(f"{len(t)} monthly measurements from {years[0]:.0f} to {years[-1]:.0f}")
print(f"Training points: {len(t_train)}, test points: {len(t_test)}")

plt.figure(figsize=(12, 3))
plt.plot(t_train, y_train, ".", ms=1, label="train")
plt.plot(t_test, y_test, ".", ms=1, label="test")
plt.xlabel("year")
plt.ylabel("temperature [°C]")
plt.legend()
plt.show()

In [ ]:
# Functions for evaluating and plotting predictions (leave this part unchanged)
#
# rmse     - root mean squared error
# coverage - fraction of observations inside the 95% interval (mean +- 2 std)
#
# plot_predictions - plots the annual means and the monthly values.
#
# The dotted line marks a warming of 1.5 °C above the mean temperature before 1980.
# mean       - the predicted temperature for every month in t_pred
# std        - (optional) the predictive standard deviation for every month in t_pred
# annual_std - (optional) the predictive standard deviation of every annual mean
#

# last year to be considered
PRED_END = 2045

# the years we will consider in the prediction plots
pred_years = np.arange(years[0], PRED_END + 1)

# collect all months until PRED_END, starting with t. The call `pred_years[:, None]' turns the pred_years vector into an n x 1 matrix.
t_pred = (pred_years[:, None] + (np.arange(12) + 0.5) / 12).ravel()

# Matrix of identity matrices for faster computation of predictions. The call np.eye generates an identity matrix 
# and np.kron applies a Kronecker product based on the provided A and B matrices (matrix with blocks of identity matrices.
A = np.kron(np.eye(len(pred_years)), np.ones(12) / 12)

# average over month
y_annual = y.reshape(-1, 12).mean(axis=1)

# visualise the 1.5 degree target that was agreed on in the Paris Agreement, adopted on 12 December 2015 at COP21.
warming_line = y_annual[years < 1980].mean() + 1.5

def plot_predictions(ax_trend, ax_zoom, title, mean, std=None, annual_std=None):
    annual_mean = A @ mean
    annual_obs, mean_obs = annual_mean[:len(years)], mean[:len(t)]  # predictions at the observed years / months
    ax_trend.plot(years[~test_years], y_annual[~test_years], ".", color="gray", label="train data")
    ax_trend.plot(years[test_years], y_annual[test_years], ".", color="black", label="test data")
    ax_trend.plot(pred_years, annual_mean, color="C0", label="prediction")
    if annual_std is not None:
        ax_trend.fill_between(pred_years, annual_mean - 2 * annual_std, annual_mean + 2 * annual_std,
                              color="C0", alpha=0.2, label="95% interval")
    ax_trend.axhline(warming_line, color="C1", ls=":", label="+1.5 °C vs. mean before 1980")
    trend_title = f"{title}: annual mean (test RMSE {rmse(annual_obs[test_years], y_annual[test_years]):.2f} °C"
    if annual_std is not None:
        trend_title += f", coverage {coverage(y_annual[test_years], annual_obs[test_years], annual_std[:len(years)][test_years]):.0%}"
    ax_trend.set_title(trend_title + ")")
    ax_trend.set_ylabel("temperature [°C]")
    ax_trend.set_ylim(TREND_YLIM)

    ax_zoom.plot(t[zoom & train], y[zoom & train], ".", color="gray", ms=3)
    ax_zoom.plot(t[zoom & ~train], y[zoom & ~train], ".", color="black", ms=3)
    ax_zoom.plot(t[zoom], mean_obs[zoom], color="C0", lw=1)
    if std is not None:
        std_obs = std[:len(t)]
        ax_zoom.fill_between(t[zoom], mean_obs[zoom] - 2 * std_obs[zoom], mean_obs[zoom] + 2 * std_obs[zoom], color="C0", alpha=0.2)
    zoom_title = f"monthly (test RMSE {rmse(mean_obs[~train], y[~train]):.2f} °C"
    if std is not None:
        zoom_title += f", coverage {coverage(y[~train], mean_obs[~train], std_obs[~train]):.0%}"
    ax_zoom.set_title(zoom_title + ")")
    ax_zoom.set_ylim(ZOOM_YLIM)

    for ax in (ax_trend, ax_zoom):
        ax.axvline(1990, color="C3", ls="--", lw=1)

## 1a. Create kernel functions

### Required: Original solution, must not have been generated with support of Generative AI

In [ ]:
# Insert the functions se_kernel, periodic_kernel, linear_kernel and combined_kernel below (directly after the comments)
#
# Input to se_kernel:
# a           - a 1-D array of n time points
# b           - a 1-D array of m time points
# sigma       - standard deviation (default = 1.0)
# lengthscale - length scale (default = 30.0)
#
# Output from se_kernel:
# K - an n x m array (Gram matrix) of the squared exponential kernel, where
#     K[i,j] = sigma^2 * exp(-(a[i] - b[j])^2 / (2 * lengthscale^2))
#
# Input to periodic_kernel:
# a           - a 1-D array of n time points
# b           - a 1-D array of m time points
# sigma       - standard deviation (default = 10.0)
# lengthscale - length scale (default = 1.0)
# period      - period (default = 1.0)
#
# Output from periodic_kernel:
# K - an n x m array (Gram matrix) of the periodic kernel, where
#     K[i,j] = sigma^2 * exp(-2 * sin^2(pi * |a[i] - b[j]| / period) / lengthscale^2)
#
# Input to linear_kernel:
# a      - a 1-D array of n time points
# b      - a 1-D array of m time points
# sigma  - standard deviation (default = 0.02)
# center - time where every line passes through zero (default = 1900.0)
#
# Output from linear_kernel:
# K - an n x m array (Gram matrix) of the linear kernel, where
#     K[i,j] = sigma^2 * (a[i] - center) * (b[j] - center)
#
# Input to combined_kernel:
# a - a 1-D array of n time points
# b - a 1-D array of m time points
#
# Output from combined_kernel:
# K - an n x m array (Gram matrix) of the sum of the squared exponential, the linear and the periodic kernel,
#     all with their default hyperparameters
#
# Hint 1: Use broadcasting; a[:, None] - b[None, :] gives the n x m array of all pairwise differences
#
# Hint 2: NumPy functions such as np.exp and np.sin work elementwise on arrays
#
# Hint 3: You can use a[:,None] to increase the dimensionality of `a' (n-dimensional vector) to an n x 1 matrix.
# 



In [ ]:
# Test your code (leave this part unchanged)

a = np.array([2000.0, 2000.25, 2000.5, 2001.0])
b = np.array([2000.0, 2030.0])

np.set_printoptions(precision=4, suppress=True)
print("se_kernel(a, b):\n{}\n".format(se_kernel(a, b)))
print("periodic_kernel(a, b):\n{}\n".format(periodic_kernel(a, b)))
print("linear_kernel(a, b):\n{}\n".format(linear_kernel(a, b)))
print("combined_kernel(a, b):\n{}".format(combined_kernel(a, b)))

### Optional: Solution obtained using Generative AI

In [ ]:
# Insert code obtained using Generative AI here
# Note that this part is not mandatory

### Required (if solution by Generative AI has been provided above): Discussion

Answer the following questions below:

- What tool/model did you use?
- How did you instruct the model to generate a solution?
- What are the main differences between your solution and the one obtained using Generative AI?
- Which solution do you think would be the most easy to understand for someone outside your group? 

### Required (if solution by Generative AI has been provided above): Combined solution

In [ ]:
# Insert the solution that you would like to use in future projects; 
# it could either be one of the two solutions above or a combination of these
# Explain in a comment why you have chosen this combination

In [ ]:
# Visualize the Gram matrices for the last 10 years of the training data (leave this part unchanged)

kernels = {"squared exponential": se_kernel,
           "periodic": periodic_kernel,
           "linear": linear_kernel,
           "combined": combined_kernel}

t_gram = t_train[t_train >= 1980]
extent = [t_gram[0], t_gram[-1], t_gram[-1], t_gram[0]]

fig, axes = plt.subplots(1, len(kernels), figsize=(5 * len(kernels), 4.5))
for ax, (name, kernel) in zip(axes, kernels.items()):
    im = ax.imshow(kernel(t_gram, t_gram), extent=extent)
    ax.set_title(name)
    ax.set_xlabel("year")
    ax.set_ylabel("year")
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

## 1b. Visualise random functions for each kernel function

### Required: Original solution, must not have been generated with support of Generative AI

In [ ]:
# Complete the code below to visualise random functions from each kernel.
#
# Draw three random functions for each kernel for the years 1900 to 2000 and show their annual means (left) and 
# their monthly function values for the last 10 years, 1990 to 1999 (right).
#
# Hint 1: Use a random seed to ensure comparability with np.random.default_rng
#
# Hint 2: To draw a random function from a kernel, evaluated at the time points t_draw, assume a vector f of function values
#         that is jointly Gaussian, f ~ N(0, K) with K = kernel(t_draw, t_draw)
#
# Hint 3: NumPy can only draw independent standard normal values, z ~ N(0, I). Use reparameterization to compute random
#         draws from a Gaussian distribution with a different mean and covariance.
#
# Hint 4: Adding a very small value (a "jitter", e.g. 1e-6) to the diagonal of K makes the matrix decomposition 
#         numerically stable.
#
# Hint 5: You can do matrix multiplication in numpy using @, i.e., f = L @ z.
#


### Optional: Solution obtained using Generative AI

In [ ]:
# Insert code obtained using Generative AI here
# Note that this part is not mandatory

### Required (if solution by Generative AI has been provided above): Discussion

Answer the following questions below:

- What tool/model did you use?
- How did you instruct the model to generate a solution?
- What are the main differences between your solution and the one obtained using Generative AI?
- Which solution do you think would be the most easy to understand for someone outside your group? 

### Required (if solution by Generative AI has been provided above): Combined solution

In [ ]:
# Insert the solution that you would like to use in future projects; 
# it could either be one of the two solutions above or a combination of these
# Explain in a comment why you have chosen this combination

### Required: Discussion

Answer the following questions below briefly (a few sentences each), referring to the figures above:

1. **Kernels and function draws**: Describe the function draws of each kernel and relate them to the Gram matrices. Why is the annual mean of every periodic draw a flat line?
2. **Linear kernel**: Is it possible to obtain the linear kernel from the squared exponential kernel? If so, how?

[Enter your answers here]

## 1c. Learn and apply kernel ridge regression

### Required: Original solution, must not have been generated with support of Generative AI

In [ ]:
# Insert the functions ridge_fit and ridge_predict below (directly after the comments)
#
# Input to ridge_fit:
# kernel - a kernel function (see 1a)
# t      - a 1-D array of n training points
# y      - a 1-D array of n (zero-mean) training targets
# lam    - regularization strength lambda (default = 4.0)
#
# Output from ridge_fit:
# alpha - a 1-D array of n dual weights
#
# Hint 1: Use np.linalg.solve to solve the linear system
#
# Hint 2: np.eye(n) gives the n x n identity matrix
#
# Input to ridge_predict:
# kernel - a kernel function (the same as used by ridge_fit)
# t      - a 1-D array of n training time points
# alpha  - the dual weights returned by ridge_fit
# t_new  - a 1-D array of m time points to predict
#
# Output from ridge_predict:
# y_new - a 1-D array of m predictions


In [ ]:
# Test your code (leave this part unchanged)

fig, axes = plt.subplots(len(kernels), 2, figsize=(16, 3.6 * len(kernels)), gridspec_kw={"width_ratios": [3, 2]})
for (ax_trend, ax_zoom), (name, kernel) in zip(axes, kernels.items()):
    alpha = ridge_fit(kernel, t_train, y_train_c)
    pred = ridge_predict(kernel, t_train, alpha, t_pred) + y_mean
    print("Kernel ridge, {}: monthly test RMSE {:.2f} °C".format(name, rmse(pred[:len(t)][~train], y_test)))
    plot_predictions(ax_trend, ax_zoom, f"Kernel ridge, {name}", pred)
axes[0, 0].legend(loc="upper left")
axes[-1, 0].set_xlabel("year")
axes[-1, 1].set_xlabel("year")
plt.tight_layout()
plt.show()

### Optional: Discussion

Answer the following questions briefly (a few sentences each), referring to the figures above:

1. **Annual vs. monthly error**: Which model would you prefer to predict the annual trend, and which would you prefer to predict the monthly trend, and why?
2. **Linear kernel**: What does the model with the linear kernel encode, and why is it helpful to include the linear kernel in the combined model?

[Enter your answers here]

### Optional: Comment on assumptions, things that do not work properly, etc.

### Optional: Solution obtained using Generative AI

In [ ]:
# Insert code obtained using Generative AI here
# Note that this part is not mandatory

### Required (if solution by Generative AI has been provided above): Discussion

Answer the following questions below:

- What tool/model did you use?
- How did you instruct the model to generate a solution?
- What are the main differences between your solution and the one obtained using Generative AI?
- Which solution do you think would be the most easy to understand for someone outside your group? 

### Required (if solution by Generative AI has been provided above): Combined solution

In [ ]:
# Insert the solution that you would like to use in future projects; 
# it could either be one of the two solutions above or a combination of these
# Explain in a comment why you have chosen this combination

## 1d. Learn and apply Gaussian process regression

### Required: Derivation

Before implementing the Gaussian process, derive how it is related to kernel ridge regression from 1b.

Let $K = k(t_{train}, t_{train})$, $K_* = k(t_{train}, t_*)$ and $K_{**} = k(t_*, t_*)$. Under the Gaussian process prior with observation noise $\sigma_n^2$, the training observations $y$ and the function values $f_*$ at new inputs $t_*$ are jointly Gaussian:

$$\begin{bmatrix} y \\ f_* \end{bmatrix} \sim \mathcal{N}\left(0, \begin{bmatrix} K + \sigma_n^2 I & K_* \\ K_*^\top & K_{**} \end{bmatrix}\right).$$

For any jointly Gaussian vector $\begin{bmatrix} a \\ b \end{bmatrix} \sim \mathcal{N}\left(0, \begin{bmatrix} \Sigma_{aa} & \Sigma_{ab} \\ \Sigma_{ba} & \Sigma_{bb} \end{bmatrix}\right)$, the conditional distribution is

$$b \mid a \sim \mathcal{N}\left(\Sigma_{ba} \Sigma_{aa}^{-1} a,\; \Sigma_{bb} - \Sigma_{ba} \Sigma_{aa}^{-1} \Sigma_{ab}\right).$$

**Questions**:
1. Derive the posterior mean $\mu_*$ and the posterior covariance $\Sigma_*$ of $f_*$ given $y$. See lecture notes for help.
2. Show that the posterior mean $\mu_*$ equals the kernel ridge prediction from 1b when $\lambda = \sigma_n^2$. 

Use the results in your implementation below: the mean of `gp_predict` should be computed by reusing `ridge_predict` from 1b.

### Put your derivation here:

1. [Enter your answers here]

2. [Enter your answers here]

### Required: Original solution, must not have been generated with support of Generative AI

In [ ]:
# Insert the functions gp_fit and gp_predict below (directly after the comments)
#
# Input to gp_fit:
# kernel      - a kernel function (see 1a)
# t           - a 1-D array of n training time points
# y           - a 1-D array of n (zero-mean) training targets
# noise_sigma - standard deviation of the observation noise (default = 2.0)
#
# Output from gp_fit:
# L     - the lower triangular Cholesky factor of K + noise_sigma^2 * I
# alpha - a 1-D array of n weights
#
# Hint 1: Use np.linalg.cholesky to compute L
#
# Hint 2: Compute alpha by solving two triangular systems, first L z = y and then L.T alpha = z
#
# Input to gp_predict:
# kernel - a kernel function (the same as used by gp_fit)
# t      - a 1-D array of n training time points
# L      - the Cholesky factor returned by gp_fit
# alpha  - the weights returned by gp_fit
# t_new  - a 1-D array of m time points to predict
#
# Output from gp_predict:
# mean - a 1-D array of m predictive means (the posterior mean)
# cov  - the m x m posterior covariance of the function values
#
# Hint 3: For the posterior mean reuse ridge_predict from 1b
#
# Hint 4: The variance of a new (noisy) observation is the diagonal of cov plus noise_sigma^2


In [ ]:
# Test your code (leave this part unchanged)
#
# Check the derivation: the GP weights are the kernel ridge weights for lam = noise_sigma^2

noise_sigma = 2.0

L, alpha = gp_fit(combined_kernel, t_train, y_train_c, noise_sigma)
alpha_ridge = ridge_fit(combined_kernel, t_train, y_train_c, lam=noise_sigma**2)
print("L is lower triangular: {}".format(np.allclose(L, np.tril(L))))
print("GP weights equal kernel ridge weights: {}".format(np.allclose(alpha, alpha_ridge)))

In [ ]:
# Test your code (leave this part unchanged)

fig, axes = plt.subplots(len(kernels), 2, figsize=(16, 3.6 * len(kernels)), gridspec_kw={"width_ratios": [3, 2]})
for (ax_trend, ax_zoom), (name, kernel) in zip(axes, kernels.items()):
    L, alpha = gp_fit(kernel, t_train, y_train_c, noise_sigma)
    mean, cov = gp_predict(kernel, t_train, L, alpha, t_pred)
    std = np.sqrt(np.diag(cov) + noise_sigma**2)
    annual_std = np.sqrt(np.diag(A @ cov @ A.T) + noise_sigma**2 / 12)
    print("GP, {}: monthly test RMSE {:.2f} °C".format(name, rmse(mean[:len(t)][~train] + y_mean, y_test)))
    plot_predictions(ax_trend, ax_zoom, f"GP, {name}", mean + y_mean, std, annual_std)
axes[0, 0].legend(loc="upper left")
axes[-1, 0].set_xlabel("year")
axes[-1, 1].set_xlabel("year")
plt.tight_layout()
plt.show()

### Optional: Discussion

Answer the following questions briefly (a few sentences each), referring to the figures above:

1. **Kernel ridge regression vs. Gaussian process**: Compare the predictions of the ridge regression and the Gaussian process. 
2. **Coverage**: For a well-calibrated model, about 95% of the test observations should lie inside the 95% interval. Which models are well calibrated monthly and/or annually? For those that aren't well-calibrated, explain why.

[Enter your answers here]

### Optional: Comment on assumptions, things that do not work properly, etc.

### Optional: Solution obtained using Generative AI

In [ ]:
# Insert code obtained using Generative AI here
# Note that this part is not mandatory

### Required (if solution by Generative AI has been provided above): Discussion

Answer the following questions below:

- What tool/model did you use?
- How did you instruct the model to generate a solution?
- What are the main differences between your solution and the one obtained using Generative AI?
- Which solution do you think would be the most easy to understand for someone outside your group? 

### Required (if solution by Generative AI has been provided above): Combined solution

In [ ]:
# Insert the solution that you would like to use in future projects; 
# it could either be one of the two solutions above or a combination of these
# Explain in a comment why you have chosen this combination

## 2a. Effect of the length scale

### Optional: Original solution, must not have been generated with support of Generative AI

In [ ]:
# Refit the GP with the combined kernel for different length scales of the squared exponential part
# and plot the predictions (directly after the comments)
#
# All hyperparameters were fixed in advance. To see what they do, use the length scales 10, 30 (default) and 100 years
# and keep all other hyperparameters at their defaults.
#
# Hint 1: Create the kernel for each length scale with a lambda function, e.g.
#         lambda a, b: se_kernel(a, b, lengthscale=lengthscale) + linear_kernel(a, b) + periodic_kernel(a, b)
#
# Hint 2: Use gp_fit, gp_predict and plot_predictions as in the test of 1c


### Optional: Discussion

Answer the following questions below briefly (a few sentences each), referring to the figures above:

1. **Length scale.** Describe how the predictions after 1990 change for $\ell_{se}$ = 10, 30 and 100 years and explain why.

[Enter your answers here]

### Optional: Comment on assumptions, things that do not work properly, etc.

### Optional: Solution obtained using Generative AI

In [ ]:
# Insert code obtained using Generative AI here
# Note that this part is not mandatory

### Required (if solution by Generative AI has been provided above): Discussion

Answer the following questions below:

- What tool/model did you use?
- How did you instruct the model to generate a solution?
- What are the main differences between your solution and the one obtained using Generative AI?
- Which solution do you think would be the most easy to understand for someone outside your group? 

### Required (if solution by Generative AI has been provided above): Combined solution

In [ ]:
# Insert the solution that you would like to use in future projects; 
# it could either be one of the two solutions above or a combination of these
# Explain in a comment why you have chosen this combination